# M6d · Build the unified ML feature view

**Goal:** materialize `marts.v_ml_features_full` — the **frozen contract**
Phase 4 modelling will consume. Combines the 35-column baseline view
(`v_ml_features_driver_behavior`) with the new ~20 archive-derived columns.

**SQL file:** `sql/26_v_ml_features_full.sql` — pure `CREATE OR REPLACE VIEW`.
No data is moved; the view recomputes on every query.

**Exit criteria:**
- `SELECT * FROM marts.v_ml_features_full LIMIT 1` succeeds.
- Column count = 35 baseline + 20 new = **55 features** + 3 identity columns.
- Spot-check: every row's `harsh_events_per_100km` is finite and ≥ 0.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs

In [2]:
from accent_fleet.db.sql_loader import load_sql
print(load_sql('26_v_ml_features_full.sql'))


-- =============================================================================
-- 26_v_ml_features_full.sql
-- =============================================================================
-- The FULL ML feature contract for Project 1 (Driver Behavior Scoring & Risk
-- Classification). Extends marts.v_ml_features_driver_behavior with the
-- archive-derived (harsh-event + telemetry) features.
--
-- Why a SEPARATE view rather than altering 22_v_ml_features.sql?
--   - The original 35-feature contract is consumed by existing notebooks /
--     baseline risk score config. Keep it stable.
--   - Adding ~20 new features doubles the column count; consumers should
--     opt-in by querying this view instead.
--   - LEFT JOIN against the new telemetry mart so devices with no archive
--     pings still produce a row (with telemetry features = 0 / NULL).
-- =============================================================================

CREATE OR REPLACE VIEW marts.v_ml_features_full AS
SELECT
  

## 3. Execute — create or replace the view

In [3]:
from accent_fleet.db import run_sql_file, transaction
with transaction() as conn:
    run_sql_file(conn, '26_v_ml_features_full.sql')
print('view created.')


view created.


## 4. Inspect — schema and sample

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    cols = pd.read_sql(text('''
        SELECT column_name, data_type
          FROM information_schema.columns
         WHERE table_schema = 'marts' AND table_name = 'v_ml_features_full'
         ORDER BY ordinal_position
    '''), conn)
    n = conn.execute(text('SELECT COUNT(*) FROM marts.v_ml_features_full')).scalar_one()
    sample = pd.read_sql(text('SELECT * FROM marts.v_ml_features_full ORDER BY harsh_events_per_100km DESC LIMIT 5'), conn)

print(f'v_ml_features_full: {len(cols)} columns, {n:,} rows')
display(cols)
display(sample)

# Sanity: harsh rates are non-negative finite
import numpy as np
arr = sample[[c for c in sample.columns if c.startswith('harsh_') and 'per_100km' in c]].to_numpy()
assert np.isfinite(arr).all() and (arr >= 0).all(), 'bad harsh-rate values'
print('OK — view ready for Phase 4 modelling.')


v_ml_features_full: 62 columns, 23,642 rows


,column_name,data_type
0,tenant_id,integer
1,device_id,bigint
2,year_month,character
3,vehicle_class_enc,integer
4,total_trips,integer
...,...,...
57,total_high_rpm_seconds,double precision
58,high_rpm_minutes_per_day,double precision
59,avg_telemetry_speed_kmh,double precision
60,max_telemetry_speed_kmh,integer


,tenant_id,device_id,year_month,vehicle_class_enc,total_trips,total_distance_km,avg_trip_distance_km,avg_trip_duration_minutes,avg_fuel_used_l,stddev_trip_distance,...,total_idle_minutes,monthly_idle_ratio,active_telemetry_days,avg_rpm,max_rpm,total_high_rpm_seconds,high_rpm_minutes_per_day,avg_telemetry_speed_kmh,max_telemetry_speed_kmh,total_fuel_used_archive
0,235,425224,2026-04,0,76,279.432405,3.676742,11.346491,0.0,5.601107,...,730.0,0.373593,10,0.0,0,0.0,0.0,6.918522,84,0.0
1,238,427661,2026-02,2,314,2597.477299,8.272221,19.173832,0.0,15.688682,...,899.0,0.094225,28,0.0,0,0.0,0.0,13.247926,84,0.0
2,238,427661,2026-04,2,117,974.715080,8.330898,20.104843,0.0,15.703833,...,579.5,0.150637,10,0.0,0,0.0,0.0,12.863083,78,0.0
3,238,427661,2026-03,2,331,2694.159462,8.139455,18.789124,0.0,16.456737,...,1264.5,0.128025,31,0.0,0,0.0,0.0,11.227026,94,0.0
4,238,427661,2026-01,2,226,1653.893036,7.318111,19.534882,0.0,15.854949,...,753.0,0.107426,31,0.0,0,0.0,0.0,10.447741,108,0.0


OK — view ready for Phase 4 modelling.
